# Mistake detection: CocoChorales-E2 and CocoChorales-E

Two audio benchmarks, with separate results and provenance:

- **CocoChorales-E2**: our monophonic injection pipeline, rendered with FluidSynth / MuseScore.
- **CocoChorales-E**: the authors’ performed MIDI and unchanged mistake labels, with both score and performance audio re-rendered using FluidSynth / `resources/MuseScore_General.sf3`. Overlapping notes remain overlapping.

**Run All only reads saved results by default.** Enable the corresponding run switch to prepare/run a benchmark in a fresh subprocess. CPU preparation is parallel; neural models use persistent, RAM-bounded worker pools. Use the Attune `at-venv` kernel and the existing [model setup instructions](../modules/mistake/README.md).

The primary measure is pooled missed/extra-event F1 at 100 ms / 50 cents; substitutions count as missed + extra. This is not note accuracy. Saved runs retain their historical code/settings. The two datasets are not paired and their scores should not be treated as a controlled comparison.

Rendering evidence and listening examples live in [coco-e_audit.ipynb](coco-e_audit.ipynb). Historical ablations are preserved in [archive](archive/mistake_experiments_2026-10-04.ipynb).

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "benchmarks/results"
RUN_E2 = False
RUN_E = False
WORKERS = max(1, (os.cpu_count() or 4) - 1)
NEURAL_WORKERS = None
RENDER_WORKERS = WORKERS
RUN_TAG = "2026-10-04"
E2_SAVED = RESULTS / "mistake_competitors_coco_unassisted_full_2026-09-29"
E_SAVED = RESULTS / "native_soundfont_2026-09-29"
E_BASE = RESULTS / "native_v5_min100_seed0_pieces3/coco"
E2_OUTPUT = RESULTS / f"coco_e2_{RUN_TAG}" if RUN_E2 else E2_SAVED
E_OUTPUT = RESULTS / f"coco_e_fluidsynth_{RUN_TAG}" if RUN_E else E_SAVED
METHODS = ("Attune (repeat only)", "PolyTune", "LadderSym")


def pooled_table(frame):
    if frame.duplicated(["method", "case_id"]).any():
        raise ValueError("Duplicate case/method rows")
    table = frame.groupby("method").agg(
        Cases=("case_id", "nunique"),
        TP=("tp", "sum"),
        FP=("fp", "sum"),
        FN=("fn", "sum"),
    )
    denominator = 2 * table.TP + table.FP + table.FN
    table["F1 %"] = 200 * table.TP / denominator.where(denominator.ne(0))
    return table.round(2)


## CocoChorales-E2 — our injection pipeline

One test stem per instrument, seeds 0 and 1, clean controls and 25% injection. Insertions/deletions shift later notes; the final performance stays monophonic. Truth is recounted after all edits. No symbolic-input baselines or refinement ablations run here.

The run cell creates fresh assets/checkpoints or resumes matching ones. Change the output tag when changing settings. Model setup/downloads are explicit prerequisites, not setup-cell side effects.

In [ ]:
if RUN_E2:
    from benchmarks.modules.mistake.datasets.MistakeCases import select_coco_sources

    sample = select_coco_sources(
        root=REPO_ROOT / "benchmarks/datasets/cocochorales_tiny",
        split="test",
        per_instrument=1,
        seed=0,
    )
    runner = [sys.executable, "-m", "benchmarks.modules.mistake.MistakeBenchmarker"]
    defaults = json.loads(
        subprocess.check_output(runner + ["--defaults"], cwd=REPO_ROOT, text=True)
    )
    request = dict(
        stage="audio",
        output=str(E2_OUTPUT),
        sources=sample.source.tolist(),
        source_metadata=sample.set_index("source").to_dict(orient="index"),
        expected_defaults=defaults,
        seeds=(0, 1),
        rates=(0.0, 0.25),
        methods=METHODS,
        tolerances=(0.05, 0.1, 0.2),
        workers=WORKERS,
        neural_workers=NEURAL_WORKERS,
        polytune_last=True,
        polytune_device="cpu",
        laddersym_device="cpu",
        force_pitch_detection=False,
    )
    subprocess.run(
        runner, input=json.dumps(request), text=True, cwd=REPO_ROOT, check=True
    )
else:
    print("Read saved E2 results:", E2_OUTPUT)


In [ ]:
if (E2_OUTPUT / "rows.csv").exists():
    metadata = json.loads((E2_OUTPUT / "run.json").read_text())
    assert (
        metadata["status"] == "complete"
    ), "Run incomplete: do not interpret partial results"
    rows = pd.read_csv(E2_OUTPUT / "rows.csv")
    primary = rows[
        rows.method.isin(METHODS)
        & rows.metric.eq("audio_pitch")
        & rows.tolerance.eq(0.1)
        & rows.input.isin(["detected", "audio"])
    ]
    display(pooled_table(primary[primary.rate.eq(0.25)]))
    display(
        primary[primary.rate.eq(0)]
        .groupby("method")
        .agg(clean_evaluations=("case_id", "nunique"), false_alarms=("fp", "sum"))
    )
    print("Recorded provenance:", E2_OUTPUT / "run.json")
else:
    print("No results yet. Enable RUN_E2 and run the preceding cell.")


## CocoChorales-E — FluidSynth + MuseScore Synthesized

The default cohort is the existing author-mirror selection: three test pieces / 12 instrument stems. `E_BASE/run.json` identifies that selection and its inference settings. Preparation checks the original performed MIDI against `correct + extra` labels, including pitch, timing, velocity and program, then re-renders both audio inputs in parallel. Labels remain byte-identical. Output is 16 kHz mono PCM24; no truth-based boundary corrections.

This preserves author polyphony, which remains challenging for Attune’s monophonic frontend. The inherited `performed_notes` pitch-range policy uses label-derived extrema (±4 semitones) and is oracle-assisted. Pretrained competitor results measure **cross-renderer generalization**. Neither a rendering repair nor a model score proves broad superiority.

The cells below do not benchmark the supplied sequentially rendered audio; that audio is retained for the dedicated audit. CPU and model pools use the worker settings above.

In [ ]:
if RUN_E:
    command = [
        sys.executable,
        "-m",
        "benchmarks.modules.mistake.MistakeBenchmarker",
        "soundfont",
    ]
    options = [
        "--base",
        str(E_BASE),
        "--output",
        str(E_OUTPUT),
        "--render-workers",
        str(RENDER_WORKERS),
        "--workers",
        str(WORKERS),
    ]
    if NEURAL_WORKERS is not None:
        options += ["--neural-workers", str(NEURAL_WORKERS)]
    subprocess.run(command + ["prepare"] + options, cwd=REPO_ROOT, check=True)
    subprocess.run(command + ["run"] + options, cwd=REPO_ROOT, check=True)
else:
    print("Read saved re-rendered E results:", E_OUTPUT)


In [ ]:
if (E_OUTPUT / "rows.csv").exists():
    metadata = json.loads((E_OUTPUT / "run.json").read_text())
    assert (
        metadata["status"] == "complete"
    ), "Run incomplete: do not interpret partial results"
    manifest = json.loads((E_OUTPUT / "manifest.json").read_text())
    rows = pd.read_csv(E_OUTPUT / "rows.csv")
    primary = rows[
        rows.protocol.eq("common_pooled_events")
        & rows.metric.eq("audio_pitch")
        & rows.tolerance.eq(0.1)
    ]
    expected = {case["case_id"] for case in manifest["cases"]}
    for method in ("Attune", "PolyTune", "LadderSym"):
        assert (
            set(primary.loc[primary.method.eq(method), "case_id"]) == expected
        ), method
    display(pooled_table(primary))
    print("Rendering contract:", manifest["intervention"])
    print("Recorded inference provenance:", E_OUTPUT / "run.json")
else:
    print("No results yet. Enable RUN_E and run the preceding cell.")
